<h1 style="text-align:center;"><b>Laboratorio 6</b></h1>
<h3 style="text-align:center;">Marcos Díaz (221102), Daniel Machic (22118), Maria Jose Ramírez (221051)</h3>

**GitHub**: https://github.com/mac2218/IA-Lab06.git

# **Problema 1**

In [7]:
import numpy as np
import time


class SudokuSolver:

    def __init__(self, n, board):
        self.n = n
        self.k = int(n**0.5)
        self.board = np.array(board)


    def is_valid(self, row, col, num):
        if num in self.board[row, :]:
            return False

        if num in self.board[:, col]:
            return False

        start_row = (row // self.k) * self.k
        start_col = (col // self.k) * self.k

        for i in range(start_row, start_row + self.k):
            for j in range(start_col, start_col + self.k):
                if self.board[i][j] == num:
                    return False

        return True


    def find_empty(self):
        for i in range(self.n):
            for j in range(self.n):
                if self.board[i][j] == 0:
                    return i, j
        return None


    def solve_backtracking(self):
        empty = self.find_empty()

        if not empty:
            return True

        row, col = empty

        for num in range(1, self.n + 1):
            if self.is_valid(row, col, num):
                self.board[row][col] = num

                if self.solve_backtracking():
                    return True

                self.board[row][col] = 0

        return False


    def init_domains(self):
        domains = {}

        for i in range(self.n):
            for j in range(self.n):
                if self.board[i][j] == 0:
                    domains[(i, j)] = set(range(1, self.n + 1))
                else:
                    domains[(i, j)] = {self.board[i][j]}

        return domains


    def propagate(self, domains, row, col, num):
        for i in range(self.n):
            if (row, i) in domains and i != col:
                if num in domains[(row, i)]:
                    domains[(row, i)].remove(num)
                    if len(domains[(row, i)]) == 0:
                        return False

            if (i, col) in domains and i != row:
                if num in domains[(i, col)]:
                    domains[(i, col)].remove(num)
                    if len(domains[(i, col)]) == 0:
                        return False

        start_row = (row // self.k) * self.k
        start_col = (col // self.k) * self.k

        for i in range(start_row, start_row + self.k):
            for j in range(start_col, start_col + self.k):
                if (i, j) in domains and (i, j) != (row, col):
                    if num in domains[(i, j)]:
                        domains[(i, j)].remove(num)
                        if len(domains[(i, j)]) == 0:
                            return False

        return True


    def solve_filtering(self):
        domains = self.init_domains()
        return self._forward_check(domains)


    def _forward_check(self, domains):
        empty = self.find_empty()

        if not empty:
            return True

        row, col = empty

        for num in list(domains[(row, col)]):
            if self.is_valid(row, col, num):

                self.board[row][col] = num

                new_domains = {k: v.copy() for k, v in domains.items()}
                new_domains[(row, col)] = {num}

                if self.propagate(new_domains, row, col, num):
                    if self._forward_check(new_domains):
                        return True

                self.board[row][col] = 0

        return False


    def display(self):
        print(self.board)


# =========================
# EJECUCIÓN
# =========================

board = [
    [5,3,0,0,7,0,0,0,0],
    [6,0,0,1,9,5,0,0,0],
    [0,9,8,0,0,0,0,6,0],
    [8,0,0,0,6,0,0,0,3],
    [4,0,0,8,0,3,0,0,1],
    [7,0,0,0,2,0,0,0,6],
    [0,6,0,0,0,0,2,8,0],
    [0,0,0,4,1,9,0,0,5],
    [0,0,0,0,8,0,0,7,9]
]


solver1 = SudokuSolver(9, board)
start = time.time()
solver1.solve_backtracking()
t1 = time.time() - start

print("Backtracking:")
solver1.display()
print("Tiempo:", t1)


solver2 = SudokuSolver(9, board)
start = time.time()
solver2.solve_filtering()
t2 = time.time() - start

print("\nForward Checking:")
solver2.display()
print("Tiempo:", t2)

Backtracking:
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]
Tiempo: 0.27471137046813965

Forward Checking:
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]
Tiempo: 0.36861324310302734


# Resolución de Sudoku con Búsqueda en Profundidad

Se implementaron dos estrategias de búsqueda en profundidad (DFS) para resolver un Sudoku de tamaño $n \times n$, donde $n = k^2$:

- Backtracking  
- Forward Checking  

---

## Estructura del Árbol de Búsqueda

El problema se modela como un árbol de búsqueda con la siguiente estructura:

- **Estado raíz:** tablero inicial con valores fijos  
- **Nodos:** tableros parcialmente llenos  
- **Hijos:** asignaciones válidas en la siguiente celda vacía  
- **Hojas:**
  - solución completa válida  
  - estado sin valores legales (poda)  

---

## Backtracking

El algoritmo de backtracking explora el espacio de soluciones asignando valores válidos a cada celda vacía.  
Cuando no es posible asignar un valor sin violar restricciones, el algoritmo retrocede al estado anterior.

---

## Forward Checking

Forward checking mejora el proceso al eliminar valores inconsistentes de las variables no asignadas después de cada asignación.  
Si algún dominio queda vacío, se descarta la rama inmediatamente.

---

## Resultados

Se ejecutaron ambas estrategias sobre el mismo Sudoku $9 \times 9$:

- **Backtracking:** 0.2689 segundos  
- **Forward Checking:** 0.3245 segundos  

Ambos métodos encontraron la misma solución.

---

## Análisis

Aunque Forward Checking reduce el espacio de búsqueda, en este caso fue ligeramente más lento debido a:

- Baja complejidad del Sudoku  
- Sobrecarga al manejar y copiar dominios  
- Mayor costo por operación en comparación con backtracking  

---

## Conclusión

Forward Checking es más eficiente en problemas más complejos, donde la poda temprana evita explorar muchas ramas.  
Sin embargo, en problemas pequeños o moderados, el overhead adicional puede hacerlo menos eficiente que el backtracking.

La elección del método depende del tamaño y dificultad del problema.

# **Problema 2**

In [8]:
import numpy as np
import time


class SudokuSolver:

    def __init__(self, n, board):
        self.n = n
        self.k = int(n**0.5)
        self.board = np.array(board)
        self.nodes_bt = 0
        self.nodes_fc = 0


    def is_valid(self, row, col, num):
        if num in self.board[row, :]:
            return False

        if num in self.board[:, col]:
            return False

        start_row = (row // self.k) * self.k
        start_col = (col // self.k) * self.k

        for i in range(start_row, start_row + self.k):
            for j in range(start_col, start_col + self.k):
                if self.board[i][j] == num:
                    return False

        return True


    def find_empty(self):
        for i in range(self.n):
            for j in range(self.n):
                if self.board[i][j] == 0:
                    return i, j
        return None


    def solve_backtracking(self):
        self.nodes_bt += 1

        empty = self.find_empty()
        if not empty:
            return True

        row, col = empty

        for num in range(1, self.n + 1):
            if self.is_valid(row, col, num):
                self.board[row][col] = num

                if self.solve_backtracking():
                    return True

                self.board[row][col] = 0

        return False


    def init_domains(self):
        domains = {}
        for i in range(self.n):
            for j in range(self.n):
                if self.board[i][j] == 0:
                    domains[(i, j)] = set(range(1, self.n + 1))
                else:
                    domains[(i, j)] = {self.board[i][j]}
        return domains


    def propagate(self, domains, row, col, num):
        for i in range(self.n):
            if (row, i) in domains and i != col:
                if num in domains[(row, i)]:
                    domains[(row, i)].remove(num)
                    if len(domains[(row, i)]) == 0:
                        return False

            if (i, col) in domains and i != row:
                if num in domains[(i, col)]:
                    domains[(i, col)].remove(num)
                    if len(domains[(i, col)]) == 0:
                        return False

        start_row = (row // self.k) * self.k
        start_col = (col // self.k) * self.k

        for i in range(start_row, start_row + self.k):
            for j in range(start_col, start_col + self.k):
                if (i, j) in domains and (i, j) != (row, col):
                    if num in domains[(i, j)]:
                        domains[(i, j)].remove(num)
                        if len(domains[(i, j)]) == 0:
                            return False

        return True


    def solve_filtering(self):
        domains = self.init_domains()
        return self._forward_check(domains)


    def _forward_check(self, domains):
        self.nodes_fc += 1

        empty = self.find_empty()
        if not empty:
            return True

        row, col = empty

        for num in list(domains[(row, col)]):
            if self.is_valid(row, col, num):

                self.board[row][col] = num

                new_domains = {k: v.copy() for k, v in domains.items()}
                new_domains[(row, col)] = {num}

                if self.propagate(new_domains, row, col, num):
                    if self._forward_check(new_domains):
                        return True

                self.board[row][col] = 0

        return False


    def display(self):
        print(self.board)

### Tableros:

In [17]:
# 4x4:
board_4 = [
    [1, 0, 0, 4],
    [0, 4, 1, 0],
    [0, 1, 4, 0],
    [2, 0, 0, 1]
]

# 9x9:
board_easy = [
    [5,3,0,0,7,0,0,0,0],
    [6,0,0,1,9,5,0,0,0],
    [0,9,8,0,0,0,0,6,0],
    [8,0,0,0,6,0,0,0,3],
    [4,0,0,8,0,3,0,0,1],
    [7,0,0,0,2,0,0,0,6],
    [0,6,0,0,0,0,2,8,0],
    [0,0,0,4,1,9,0,0,5],
    [0,0,0,0,8,0,0,7,9]
]

# 9x9 extremo:

board_hard = [
    [0,0,0,0,0,0,0,1,2],
    [0,0,0,0,0,0,0,3,0],
    [0,0,1,0,9,4,5,0,0],
    [0,0,0,0,0,0,0,0,0],
    [0,0,0,5,3,8,0,0,0],
    [0,0,0,0,0,0,0,0,0],
    [0,0,9,7,1,0,3,0,0],
    [0,4,0,0,0,0,0,0,0],
    [3,7,0,0,0,0,0,0,0]
]

#16x16:

board_16 = [[0]*16 for _ in range(16)]

In [18]:
def run_case(name, n, board):
    print("\n==========", name, "==========")

    solver_bt = SudokuSolver(n, board)
    start = time.time()
    solver_bt.solve_backtracking()
    t_bt = time.time() - start

    print("\nBacktracking:")
    solver_bt.display()
    print("Tiempo:", t_bt)
    print("Nodos:", solver_bt.nodes_bt)

    solver_fc = SudokuSolver(n, board)
    start = time.time()
    solver_fc.solve_filtering()
    t_fc = time.time() - start

    print("\nForward Checking:")
    solver_fc.display()
    print("Tiempo:", t_fc)
    print("Nodos:", solver_fc.nodes_fc)


run_case("4x4", 4, board_4)
run_case("9x9 Fácil", 9, board_easy)
run_case("9x9 Extremo", 9, board_hard)
# run_case("16x16", 16, board_16)  # cuidado: puede tardar mucho


========== 4x4 ==========

Backtracking:
[[1 0 0 4]
 [0 4 1 0]
 [0 1 4 0]
 [2 0 0 1]]
Tiempo: 0.00033402442932128906
Nodos: 7

Forward Checking:
[[1 0 0 4]
 [0 4 1 0]
 [0 1 4 0]
 [2 0 0 1]]
Tiempo: 0.00025153160095214844
Nodos: 7

========== 9x9 Fácil ==========

Backtracking:
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]
Tiempo: 0.3021059036254883
Nodos: 4209

Forward Checking:
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]
Tiempo: 0.5735189914703369
Nodos: 4209

========== 9x9 Extremo ==========

Backtracking:
[[4 3 5 6 8 7 9 1 2]
 [6 9 7 1 2 5 4 3 8]
 [2 8 1 3 9 4 5 6 7]
 [5 1 3 2 6 9 8 7 4]
 [7 2 4 5 3 8 6 9 1]
 [9 6 8 4 7 1 2 5 3]
 [8 5 9 7 1 2 3 4 6]
 [1 4 6 8 5 3 7 2 9]
 [3 7 2 9 4 6 1 8 5]]
Tiem

A partir de los resultados obtenidos, no hay un ganador absoluto en todos los casos, pero sí se observa una tendencia clara:

En el Sudoku 4x4, ambos métodos tienen prácticamente el mismo rendimiento, ya que el problema es demasiado pequeño para notar diferencias.
En el 9x9 fácil, el forward checking resulta más lento que el backtracking y explora la misma cantidad de nodos, lo que indica que el costo adicional de mantener y actualizar dominios no se compensa en problemas simples.
En el 9x9 extremo, el forward checking sí muestra su ventaja: reduce el número de nodos explorados (5970 vs 6616) y también el tiempo de ejecución, al anticipar inconsistencias y podar el árbol antes.

**Conclusión:**
El backtracking es más eficiente en problemas pequeños o poco complejos debido a su menor sobrecarga. Sin embargo, el forward checking se vuelve más eficiente en problemas más difíciles, donde la poda temprana del espacio de búsqueda reduce significativamente el trabajo total.

# **Problema 3**

In [ ]:
import time

class NQueensSolver:
    def __init__(self, n):
        self.n = n
        self.board = [-1] * n  # a[i] = columna
        self.nodes = 0

    def is_valid(self, row, col):
        for i in range(row):
            if self.board[i] == col or abs(self.board[i] - col) == abs(i - row):
                return False
        return True

    def solve_backtracking(self, row=0):
        if row == self.n:
            return True

        for col in range(self.n):
            self.nodes += 1
            if self.is_valid(row, col):
                self.board[row] = col
                if self.solve_backtracking(row + 1):
                    return True
                self.board[row] = -1

        return False

    def solve_filtering(self, row=0, domains=None):
        if domains is None:
            domains = [list(range(self.n)) for _ in range(self.n)]

        if row == self.n:
            return True

        for col in domains[row]:
            self.nodes += 1
            if self.is_valid(row, col):
                self.board[row] = col

                # Copia de dominios
                new_domains = [d.copy() for d in domains]

                # Forward Checking: eliminar valores inválidos
                for r in range(row + 1, self.n):
                    new_domains[r] = [
                        c for c in new_domains[r]
                        if c != col and abs(c - col) != abs(r - row)
                    ]

                    # Si algún dominio queda vacío → poda
                    if not new_domains[r]:
                        break
                else:
                    if self.solve_filtering(row + 1, new_domains):
                        return True

                self.board[row] = -1

        return False

    def display(self):
        for i in range(self.n):
            row = ['.'] * self.n
            if self.board[i] != -1:
                row[self.board[i]] = 'Q'
            print(' '.join(row))
        print()


In [ ]:
def run_solver(n):
    print(f"\n=========== N = {n} ===========")

    # BACKTRACKING
    solver_bt = NQueensSolver(n)
    start = time.time()
    solver_bt.solve_backtracking()
    end = time.time()

    print("\nBacktracking:")
    solver_bt.display()
    print("Nodos visitados:", solver_bt.nodes)
    print("Tiempo:", end - start)

    # FORWARD CHECKING
    solver_fc = NQueensSolver(n)
    start = time.time()
    solver_fc.solve_filtering()
    end = time.time()

    print("\nForward Checking:")
    solver_fc.display()
    print("Nodos visitados:", solver_fc.nodes)
    print("Tiempo:", end - start)



# **Problema 4**

In [ ]:
run_solver(4)


=========== N = 4 ===========

Backtracking:
. Q . .
. . . Q
Q . . .
. . Q .

Nodos visitados: 26
Tiempo: 0.0

Forward Checking:
. Q . .
. . . Q
Q . . .
. . Q .

Nodos visitados: 8
Tiempo: 0.0


In [ ]:
run_solver(8)


=========== N = 8 ===========

Backtracking:
Q . . . . . . .
. . . . Q . . .
. . . . . . . Q
. . . . . Q . .
. . Q . . . . .
. . . . . . Q .
. Q . . . . . .
. . . Q . . . .

Nodos visitados: 876
Tiempo: 0.0010004043579101562

Forward Checking:
Q . . . . . . .
. . . . Q . . .
. . . . . . . Q
. . . . . Q . .
. . Q . . . . .
. . . . . . Q .
. Q . . . . . .
. . . Q . . . .

Nodos visitados: 88
Tiempo: 0.0010004043579101562


In [ ]:
run_solver(12)


=========== N = 12 ===========

Backtracking:
Q . . . . . . . . . . .
. . Q . . . . . . . . .
. . . . Q . . . . . . .
. . . . . . . Q . . . .
. . . . . . . . . Q . .
. . . . . . . . . . . Q
. . . . . Q . . . . . .
. . . . . . . . . . Q .
. Q . . . . . . . . . .
. . . . . . Q . . . . .
. . . . . . . . Q . . .
. . . Q . . . . . . . .

Nodos visitados: 3066
Tiempo: 0.001978635787963867

Forward Checking:
Q . . . . . . . . . . .
. . Q . . . . . . . . .
. . . . Q . . . . . . .
. . . . . . . Q . . . .
. . . . . . . . . Q . .
. . . . . . . . . . . Q
. . . . . Q . . . . . .
. . . . . . . . . . Q .
. Q . . . . . . . . . .
. . . . . . Q . . . . .
. . . . . . . . Q . . .
. . . Q . . . . . . . .

Nodos visitados: 193
Tiempo: 0.0009996891021728516


In [ ]:
run_solver(15)


=========== N = 15 ===========

Backtracking:
Q . . . . . . . . . . . . . .
. . Q . . . . . . . . . . . .
. . . . Q . . . . . . . . . .
. Q . . . . . . . . . . . . .
. . . . . . . . . Q . . . . .
. . . . . . . . . . . Q . . .
. . . . . . . . . . . . . Q .
. . . Q . . . . . . . . . . .
. . . . . . . . . . . . Q . .
. . . . . . . . Q . . . . . .
. . . . . Q . . . . . . . . .
. . . . . . . . . . . . . . Q
. . . . . . Q . . . . . . . .
. . . . . . . . . . Q . . . .
. . . . . . . Q . . . . . . .

Nodos visitados: 20280
Tiempo: 0.027281761169433594

Forward Checking:
Q . . . . . . . . . . . . . .
. . Q . . . . . . . . . . . .
. . . . Q . . . . . . . . . .
. Q . . . . . . . . . . . . .
. . . . . . . . . Q . . . . .
. . . . . . . . . . . Q . . .
. . . . . . . . . . . . . Q .
. . . Q . . . . . . . . . . .
. . . . . . . . . . . . Q . .
. . . . . . . . Q . . . . . .
. . . . . Q . . . . . . . . .
. . . . . . . . . . . . . . Q
. . . . . . Q . . . . . . . .
. . . . . . . . . . Q . . . .
. . . . . .

Los resultados obtenidos muestran que el algoritmo de Forward Checking es considerablemente más eficiente que el Backtracking puro, especialmente a medida que aumenta el tamaño del problema. Aunque en casos pequeños como N=4 la diferencia en tiempo es mínima e incluso puede existir un ligero overhead debido al manejo de dominios, la ventaja de Forward Checking se vuelve evidente en instancias más grandes. Esto se refleja claramente en la reducción significativa del número de nodos explorados: por ejemplo, para N=15, Backtracking visita 20280 nodos mientras que Forward Checking solo 1026, lo que representa una disminución de aproximadamente 20 veces. Esta reducción se traduce también en menores tiempos de ejecución. La razón principal es que Forward Checking aplica poda temprana al eliminar valores inconsistentes de las variables no asignadas, evitando así explorar ramas inviables del árbol de búsqueda. En contraste, Backtracking detecta los conflictos de manera tardía, lo que lo obliga a recorrer una mayor cantidad de estados. En consecuencia, se concluye que Forward Checking escala mejor y es más adecuado para resolver problemas de mayor tamaño dentro del contexto de CSP.

# **Problema 5**

In [ ]:
import time

class NQueensSolver:
    def __init__(self, n):
        self.n = n
        self.board = [-1] * n
        self.nodes = 0
        self.solutions = []

    def is_valid(self, row, col):
        for i in range(row):
            if self.board[i] == col or abs(self.board[i] - col) == abs(i - row):
                return False
        return True

    def solve_all(self, row=0):
        if row == self.n:
            self.solutions.append(self.board.copy())
            return

        for col in range(self.n):
            if self.is_valid(row, col):
                self.board[row] = col
                self.solve_all(row + 1)
                self.board[row] = -1

    def display_solutions(self):
        for idx, sol in enumerate(self.solutions):
            print(f"\nSolución {idx + 1}:")
            for i in range(self.n):
                row = ['.'] * self.n
                row[sol[i]] = 'Q'
                print(' '.join(row))

def run_all_solutions(n, show=False):
    solver = NQueensSolver(n)
    solver.solve_all()

    print(f"\nN = {n}")
    print("Número de soluciones:", len(solver.solutions))

    if show:
        solver.display_solutions()

In [ ]:
run_all_solutions(4, True)


N = 4
Número de soluciones: 2

Solución 1:
. Q . .
. . . Q
Q . . .
. . Q .

Solución 2:
. . Q .
Q . . .
. . . Q
. Q . .


In [ ]:
run_all_solutions(5)



N = 5
Número de soluciones: 10


In [ ]:
run_all_solutions(6)


N = 6
Número de soluciones: 4


Para obtener todas las soluciones del problema de N reinas, se modificó el algoritmo de backtracking eliminando la condición de parada temprana, de modo que el algoritmo no se detiene al encontrar una solución válida, sino que continúa explorando todo el árbol de búsqueda. Cada vez que se alcanza un estado completo válido, se almacena una copia del tablero en una lista de soluciones. Esto permite enumerar todas las configuraciones posibles que cumplen las restricciones del problema. Los resultados obtenidos para N=4,5,6 fueron 2, 10 y 4 soluciones respectivamente, coincidiendo con los valores teóricos conocidos, lo que valida la correcta implementación del algoritmo.

#### **Pregunta de Discusión**
El enfoque de Forward Checking es más eficiente que el Backtracking puro, especialmente a medida que aumenta el tamaño del problema. Esto se debe a que el backtracking explora el árbol de búsqueda sin anticipar conflictos, detectando inconsistencias únicamente cuando ya ha profundizado en ramas inválidas, lo que provoca un mayor número de nodos explorados. En contraste, el forward checking aplica un mecanismo de filtrado que reduce dinámicamente los dominios de las variables no asignadas, eliminando valores que violan restricciones antes de continuar la búsqueda. Esto permite realizar poda temprana del árbol, evitando explorar estados inviables. Como se observó en los resultados experimentales, el número de nodos visitados y el tiempo de ejecución son significativamente menores con forward checking, especialmente en valores grandes de N, lo que demuestra que este enfoque escala mejor y es computacionalmente más eficiente.